# Task 1: Model Training and Optimization Pipeline
Preprocessing, hyperparameter tuning via 5-fold CV (Grid / Random / Bayesian), experiment tracking with trackio, and final evaluation on the test set.

In [1]:
!pip3 install --quiet pandas numpy matplotlib scikit-learn optuna trackio

In [2]:
import os
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
import trackio

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Create required output folders
os.makedirs('models', exist_ok=True)
os.makedirs('plots', exist_ok=True)
os.makedirs('screenshots', exist_ok=True)

RANDOM_STATE = 42
CV_FOLDS = 5
SCORING = 'neg_mean_absolute_error'  # final metric is MAE — keep CV scoring consistent

## 1. Data Loading & Preprocessing
Load the **updated** `train.csv` and `test.csv`. Convert string categorical variables to numeric using `LabelEncoder`. Save encoders for the Streamlit UI.

In [3]:
# NOTE: TA pushed updated CSVs. Adjust path to wherever you've placed them.
# Repo layout per assignment: Dataset/train.csv  and  Dataset/test.csv
train_df = pd.read_csv('Dataset/train.csv')
test_df  = pd.read_csv('Dataset/test.csv')
train_df = train_df.drop(columns=['Price_per_sqft'])
test_df  = test_df.drop(columns=['Price_per_sqft'])

print('Train shape:', train_df.shape)
print('Test  shape:', test_df.shape)
print('Columns:', list(train_df.columns))

Train shape: (11128, 15)
Test  shape: (2782, 15)
Columns: ['location', 'city', 'latitude', 'longitude', 'price', 'numBathrooms', 'numBalconies', 'isNegotiable', 'SecurityDeposit', 'Status', 'Size_ft²', 'BHK', 'rooms_num', 'property_type', 'verification_days']


In [4]:
# Encode categorical columns using LabelEncoder fit on combined train+test
# so unseen categories at inference time don't break the encoder.
categorical_columns = train_df.select_dtypes(include=['object']).columns.tolist()
print('Categorical columns:', categorical_columns)

encoders = {}
for col in categorical_columns:
    le = LabelEncoder()
    combined = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
    le.fit(combined)
    train_df[col] = le.transform(train_df[col].astype(str))
    test_df[col]  = le.transform(test_df[col].astype(str))
    encoders[col] = le

# Save encoders for the Streamlit UI to reuse
with open('models/label_encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
print('Saved models/label_encoders.pkl')

# Separate predictors and target
X_train = train_df.drop('price', axis=1)
y_train = train_df['price']
X_test  = test_df.drop('price', axis=1)
y_test  = test_df['price']

# Save the feature column order — Streamlit must pass features in the same order
with open('models/feature_columns.pkl', 'wb') as f:
    pickle.dump(list(X_train.columns), f)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

Categorical columns: ['location', 'city', 'Status', 'property_type']
Saved models/label_encoders.pkl
X_train: (11128, 14), X_test: (2782, 14)


## 2. Hyperparameter Tuning (5-fold Cross-Validation)

**Grid Search — exactly 60 combinations:**
- `n_estimators`: [50, 100, 150, 200]   (4)
- `max_depth`:    [10, 15, 20, 25, 30]  (5)
- `min_samples_split`: [2, 5, 8]        (3)

**Random Search & Bayesian (Optuna):**
- `n_estimators`: int 50–200
- `max_depth`: int 10–30
- `min_samples_split`: int 2–10
- Budget: 60 iterations each (within 60–100 allowed range)

In [5]:

# Helper: cast numpy types to plain Python so trackio can JSON-serialize
def to_py(x):
    if isinstance(x, (np.floating,)): return float(x)
    if isinstance(x, (np.integer,)):  return int(x)
    return x

BUDGET = 60  # iterations for Random and Bayesian (assignment allows 60-100)

In [6]:
# ---------- 1) GRID SEARCH (exactly 60 combinations) ----------
trackio.init(project='Price_Prediction', name='GridSearch')
print('--- Grid Search ---')
param_grid = {
    'n_estimators':      [50, 100, 150, 200],   # 4
    'max_depth':         [10, 15, 20, 25, 30],  # 5
    'min_samples_split': [2, 5, 8],             # 3
}
assert 4*5*3 == 60, 'Grid must be exactly 60 combinations'

rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)
grid_search = GridSearchCV(
    estimator=rf, param_grid=param_grid,
    cv=CV_FOLDS, scoring=SCORING, n_jobs=-1, return_train_score=False,
)

t0 = time.time()
grid_search.fit(X_train, y_train)
grid_time = time.time() - t0
grid_iters = len(grid_search.cv_results_['params'])
grid_best = float(grid_search.best_score_)  # neg MAE (closer to 0 better)

print(f'Grid: best CV neg-MAE = {grid_best:.2f} | iters={grid_iters} | time={grid_time:.1f}s')
print('Best params:', grid_search.best_params_)

trackio.log({
    'method': 'GridSearch',
    'time_seconds': float(grid_time),
    'iterations': int(grid_iters),
    'best_cv_neg_mae': grid_best,
    'best_n_estimators': int(grid_search.best_params_['n_estimators']),
    'best_max_depth': int(grid_search.best_params_['max_depth']),
    'best_min_samples_split': int(grid_search.best_params_['min_samples_split']),
})

trackio.finish()

* Trackio project initialized: Price_Prediction
* Trackio metrics logged to: /Users/adityapriyadarshi/.cache/huggingface/trackio
* Apple Silicon detected, enabling automatic system metrics logging
* Created new run: GridSearch


--- Grid Search ---


/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyad

Grid: best CV neg-MAE = -13276.83 | iters=60 | time=322.3s
Best params: {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}
* Run finished. Uploading logs to Trackio (please wait...)


In [7]:
# ---------- 2) RANDOM SEARCH (60 iterations) ----------
trackio.init(project='Price_Prediction', name='RandomSearch')
print('--- Random Search ---')
param_distributions = {
    'n_estimators':      list(range(50, 201)),
    'max_depth':         list(range(10, 31)),
    'min_samples_split': list(range(2, 11)),
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=param_distributions,
    n_iter=BUDGET, cv=CV_FOLDS, scoring=SCORING,
    random_state=RANDOM_STATE, n_jobs=-1,
)

t0 = time.time()
random_search.fit(X_train, y_train)
random_time = time.time() - t0
random_best = float(random_search.best_score_)

print(f'Random: best CV neg-MAE = {random_best:.2f} | iters={BUDGET} | time={random_time:.1f}s')
print('Best params:', random_search.best_params_)

trackio.log({
    'method': 'RandomSearch',
    'time_seconds': float(random_time),
    'iterations': int(BUDGET),
    'best_cv_neg_mae': random_best,
    'best_n_estimators': int(random_search.best_params_['n_estimators']),
    'best_max_depth': int(random_search.best_params_['max_depth']),
    'best_min_samples_split': int(random_search.best_params_['min_samples_split']),
})

trackio.finish()

* Apple Silicon detected, enabling automatic system metrics logging
* Created new run: RandomSearch
--- Random Search ---


/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyad

Random: best CV neg-MAE = -13307.13 | iters=60 | time=217.6s
Best params: {'n_estimators': 142, 'min_samples_split': 2, 'max_depth': 24}
* Run finished. Uploading logs to Trackio (please wait...)


In [8]:
# ---------- 3) BAYESIAN OPTIMIZATION (Optuna, 60 trials) ----------
trackio.init(project='Price_Prediction', name='BayesianOptuna')
print('--- Bayesian Optimization (Optuna) ---')

cv_splitter = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 200),
        'max_depth':         trial.suggest_int('max_depth', 10, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
    }
    model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    score = cross_val_score(model, X_train, y_train, cv=cv_splitter,
                            scoring=SCORING, n_jobs=-1).mean()
    return score  # neg MAE — maximize

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=RANDOM_STATE),
)

t0 = time.time()
study.optimize(objective, n_trials=BUDGET, show_progress_bar=False)
optuna_time = time.time() - t0
optuna_best = float(study.best_value)

print(f'Optuna: best CV neg-MAE = {optuna_best:.2f} | trials={BUDGET} | time={optuna_time:.1f}s')
print('Best params:', study.best_params)

trackio.log({
    'method': 'BayesianOptuna',
    'time_seconds': float(optuna_time),
    'iterations': int(BUDGET),
    'best_cv_neg_mae': optuna_best,
    'best_n_estimators': int(study.best_params['n_estimators']),
    'best_max_depth': int(study.best_params['max_depth']),
    'best_min_samples_split': int(study.best_params['min_samples_split']),
})

trackio.finish()

* Apple Silicon detected, enabling automatic system metrics logging


[I 2026-04-15 01:26:25,312] A new study created in memory with name: no-name-aa29a50c-722e-4e8d-b9c2-668ba07bed46


* Created new run: BayesianOptuna
--- Bayesian Optimization (Optuna) ---


/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/sklearn/utils/parallel.py:144: UserWarning: `sklearn.utils.parallel.delayed` should be used with `sklearn.utils.parallel.Parallel` to make it possible to propagate the scikit-learn configuration of the current thread to the joblib workers.
  warnings.warn(
/Users/adityapriyad

Optuna: best CV neg-MAE = -13495.13 | trials=60 | time=386.2s
Best params: {'n_estimators': 155, 'max_depth': 24, 'min_samples_split': 2}
* Run finished. Uploading logs to Trackio (please wait...)


## 3. Evaluation & Plots
Compare the three strategies on a single trials-vs-error plot, and visualize Optuna's exploration of the hyperparameter space.

In [9]:
# ---------- Plot 1: trials_vs_error.png ----------
# Convert neg-MAE -> MAE (positive). Use cumulative best (running min).
grid_errs   = -np.array(grid_search.cv_results_['mean_test_score'])
random_errs = -np.array(random_search.cv_results_['mean_test_score'])
optuna_errs = np.array([-t.value for t in study.trials])

grid_cum   = np.minimum.accumulate(grid_errs)
random_cum = np.minimum.accumulate(random_errs)
optuna_cum = np.minimum.accumulate(optuna_errs)

plt.figure(figsize=(10, 6))
plt.plot(range(1, len(grid_cum)+1),   grid_cum,   marker='o', ms=3, label='Grid Search')
plt.plot(range(1, len(random_cum)+1), random_cum, marker='s', ms=3, label='Random Search')
plt.plot(range(1, len(optuna_cum)+1), optuna_cum, marker='^', ms=3, label='Bayesian (Optuna)')
plt.xlabel('Iteration / Trial #')
plt.ylabel('Best CV MAE found so far')
plt.title('Compute Budget vs. Cross-Validation Error')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig('plots/trials_vs_error.png', dpi=150)
plt.show()
print('Saved plots/trials_vs_error.png')

Saved plots/trials_vs_error.png


/var/folders/k1/5m1wklwj1x36v_21zz0f_jdr0000gn/T/ipykernel_70629/2875609602.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
# ---------- Plot 2: optuna_hyperparameter_space.png ----------
# Use Optuna's built-in matplotlib visualization (assignment recommends this).
from optuna.visualization.matplotlib import plot_optimization_history, plot_contour

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

plt.sca(axes[0])
plot_optimization_history(study)
axes[0].set_title('Optuna Optimization History')

plt.sca(axes[1])
plot_contour(study, params=['n_estimators', 'max_depth'])
axes[1].set_title('Optuna Contour: n_estimators vs max_depth')

plt.tight_layout()
plt.savefig('plots/optuna_hyperparameter_space.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved plots/optuna_hyperparameter_space.png')

/var/folders/k1/5m1wklwj1x36v_21zz0f_jdr0000gn/T/ipykernel_70629/3899423026.py:8: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  plot_optimization_history(study)
/var/folders/k1/5m1wklwj1x36v_21zz0f_jdr0000gn/T/ipykernel_70629/3899423026.py:12: ExperimentalWarning: optuna.visualization.matplotlib._contour.plot_contour is experimental (supported from v2.2.0). The interface can change in the future.
  plot_contour(study, params=['n_estimators', 'max_depth'])


Saved plots/optuna_hyperparameter_space.png


/var/folders/k1/5m1wklwj1x36v_21zz0f_jdr0000gn/T/ipykernel_70629/3899423026.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Final Testing & Model Saving
Report best hyperparameters from each method, retrain the overall best on the full training set, evaluate on the held-out test set (MAE), and save artifacts.

In [11]:
print('='*60)
print('Best Hyperparameters by Method (CV scoring = neg MAE)')
print('='*60)
print(f'Grid Search    : {grid_search.best_params_}  | best CV MAE = {-grid_best:.2f}')
print(f'Random Search  : {random_search.best_params_}  | best CV MAE = {-random_best:.2f}')
print(f'Bayesian Optuna: {study.best_params}  | best CV MAE = {-optuna_best:.2f}')

# Pick overall best (max neg-MAE = min MAE)
candidates = {
    'GridSearch':    (grid_search.best_params_, grid_best),
    'RandomSearch':  (random_search.best_params_, random_best),
    'BayesianOptuna':(study.best_params, optuna_best),
}
best_method = max(candidates, key=lambda k: candidates[k][1])
best_params = candidates[best_method][0]
print(f'\n>>> Overall best method: {best_method}')
print(f'>>> Best params: {best_params}')

Best Hyperparameters by Method (CV scoring = neg MAE)
Grid Search    : {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}  | best CV MAE = 13276.83
Random Search  : {'n_estimators': 142, 'min_samples_split': 2, 'max_depth': 24}  | best CV MAE = 13307.13
Bayesian Optuna: {'n_estimators': 155, 'max_depth': 24, 'min_samples_split': 2}  | best CV MAE = 13495.13

>>> Overall best method: GridSearch
>>> Best params: {'max_depth': 25, 'min_samples_split': 2, 'n_estimators': 200}


In [12]:
# Retrain on full training set and evaluate on test set
best_model = RandomForestRegressor(
    random_state=RANDOM_STATE, n_jobs=-1, **best_params
)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
test_mae = mean_absolute_error(y_test, y_pred)
print(f'Final Test MAE: {test_mae:.2f}')

trackio.log({
    'method': 'FinalModel',
    'best_method': best_method,
    'test_mae': float(test_mae),
})

# Save model + encoders to models/ folder
with open('models/best_rf_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print('Saved models/best_rf_model.pkl')

# (label_encoders.pkl + feature_columns.pkl already saved in section 1)
trackio.finish()

Final Test MAE: 12420.35
Saved models/best_rf_model.pkl
* Run finished. Uploading logs to Trackio (please wait...)


/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/gradio/routes.py:1390: DeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(full_body, request, username)
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/gradio/utils.py:1011: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Python 3.16; use inspect.iscoroutinefunction() instead
  elif asyncio.iscoroutinefunction(f):
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/gradio/routes.py:1390: DeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(full_body, request, username)
/Users/adityapriyadarshi/Library/Python/3.14/lib/python/site-packages/gradio/utils.py:1011: DeprecationWarning: 'asyncio.iscoroutinefunction' is deprecated and slated for removal in Pyth